# Retail Sales Data Analysis & Business Insights

**Slab 1 – For Beginners**

This notebook is designed as a complete, reproducible submission. Run the cells from top to bottom.

## 1. Objective
Analyze retail sales data using Pandas, Matplotlib and Seaborn. The analysis covers data quality, sales/profit/quantity/customer statistics, product/category/customer/regional performance, trends, relationships and business recommendations.

**Dataset:** Sample Superstore, a commonly used retail transaction dataset containing order, customer, product, geography, sales, quantity, discount and profit information.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

# Public mirror of the Sample Superstore CSV.
DATA_URL = "https://raw.githubusercontent.com/leonism/sample-superstore/master/data/superstore.csv"

df = pd.read_csv(DATA_URL)
print("Shape:", df.shape)
display(df.head())

## 2. Data cleaning and validation
We standardize column names, convert dates and numeric fields, inspect missing values and duplicates, and remove records that cannot support meaningful sales analysis.

In [ ]:
df.columns = (
    df.columns.str.strip()
              .str.lower()
              .str.replace(" ", "_", regex=False)
)

# Convert common date/numeric columns when present.
for col in ["order_date", "ship_date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

for col in ["sales", "profit", "quantity", "discount"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("Missing values:")
display(df.isna().sum().sort_values(ascending=False).head(15))

print("Duplicate rows:", df.duplicated().sum())

# Remove exact duplicates.
df = df.drop_duplicates().copy()

# Basic cleaning for records needed in KPI analysis.
required = [c for c in ["sales", "profit", "quantity"] if c in df.columns]
df = df.dropna(subset=required)

print("Cleaned shape:", df.shape)

## 3. Feature engineering and descriptive statistics
We create profit margin and calendar features to make the business analysis more informative.

In [ ]:
df["profit_margin_pct"] = np.where(
    df["sales"] != 0, df["profit"] / df["sales"] * 100, np.nan
)

if "order_date" in df.columns:
    df["year"] = df["order_date"].dt.year
    df["month"] = df["order_date"].dt.month
    df["month_name"] = df["order_date"].dt.month_name()
    df["quarter"] = df["order_date"].dt.quarter

kpis = pd.Series({
    "Total Sales": df["sales"].sum(),
    "Total Profit": df["profit"].sum(),
    "Total Quantity": df["quantity"].sum(),
    "Average Order/Row Sales": df["sales"].mean(),
    "Average Profit/Row": df["profit"].mean(),
    "Profit Margin %": df["profit"].sum() / df["sales"].sum() * 100
})
display(kpis.to_frame("Value").round(2))

display(df[["sales","profit","quantity","discount","profit_margin_pct"]].describe().T.round(2))

## 4. Exploratory data analysis
The following visualizations satisfy the beginner-level requirement for bar charts, line charts, scatter plots and a heatmap.

In [ ]:
# Sales and profit by category
cat = df.groupby("category")[["sales","profit"]].sum().sort_values("sales", ascending=False)
cat.plot(kind="bar", figsize=(10,5), title="Sales and Profit by Category")
plt.ylabel("Amount")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Monthly sales trend
monthly = df.set_index("order_date").resample("M")["sales"].sum()
monthly.plot(figsize=(12,5), marker="o", title="Monthly Sales Trend")
plt.ylabel("Sales")
plt.tight_layout()
plt.show()

# Sales vs profit
plt.figure(figsize=(9,6))
sns.scatterplot(data=df.sample(min(3000, len(df)), random_state=42),
                x="sales", y="profit", hue="category", alpha=0.65)
plt.title("Sales vs Profit")
plt.tight_layout()
plt.show()

# Numeric correlation heatmap
num_cols = df.select_dtypes(include=np.number).columns
plt.figure(figsize=(10,7))
sns.heatmap(df[num_cols].corr(), cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## 5. Product, customer and regional performance

In [ ]:
# Top products by sales
if "product_name" in df.columns:
    top_products = df.groupby("product_name")["sales"].sum().nlargest(10)
    display(top_products.to_frame("Total Sales"))
    top_products.sort_values().plot(kind="barh", figsize=(9,6), title="Top 10 Products by Sales")
    plt.xlabel("Sales")
    plt.tight_layout()
    plt.show()

# Customer performance
if "customer_name" in df.columns:
    top_customers = df.groupby("customer_name").agg(
        total_sales=("sales","sum"),
        total_profit=("profit","sum"),
        orders=("order_id","nunique") if "order_id" in df.columns else ("sales","size")
    ).sort_values("total_sales", ascending=False).head(10)
    display(top_customers.round(2))

# Regional performance
region = df.groupby("region").agg(
    sales=("sales","sum"),
    profit=("profit","sum"),
    quantity=("quantity","sum")
).sort_values("sales", ascending=False)
region["profit_margin_pct"] = region["profit"] / region["sales"] * 100
display(region.round(2))

region[["sales","profit"]].plot(kind="bar", figsize=(10,5), title="Regional Sales and Profit")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. Discount and profitability analysis
A high sales figure does not automatically mean a healthy business. This section checks whether discounting is associated with lower profitability.

In [ ]:
discount_analysis = (
    df.groupby("discount")
      .agg(avg_sales=("sales","mean"), avg_profit=("profit","mean"),
           total_sales=("sales","sum"), total_profit=("profit","sum"), rows=("sales","size"))
      .sort_index()
)
display(discount_analysis.round(2))

plt.figure(figsize=(9,6))
sns.scatterplot(data=df.sample(min(3000, len(df)), random_state=1),
                x="discount", y="profit", alpha=0.5)
plt.title("Discount vs Profit")
plt.tight_layout()
plt.show()

loss_makers = (
    df.groupby("sub_category")["profit"].sum()
      .sort_values()
      .head(10)
)
display(loss_makers.to_frame("Total Profit"))

## 7. Business insights and recommendations
Use the actual tables and charts above to write the final observations. The following framework can be used in the report:

- Prioritize categories and regions with both high revenue and healthy profit margins.
- Investigate products/sub-categories with repeated negative profit.
- Avoid evaluating performance on sales alone; include profit margin.
- Review high-discount transactions where profitability deteriorates.
- Retain high-value customers with personalized offers and service.
- Prepare inventory and marketing plans around visible monthly/quarterly demand patterns.

**Important:** Do not copy generic statements as facts. Replace them with the specific categories, regions and products shown by the executed notebook.